# Code review: сломанный PyTorch train loop

ML Mentor · starter notebook · решения в репозитории отсутствуют.


## Задача

Исправьте цикл ниже. В нём намеренно смешаны ошибки управления градиентами, режимов модели и validation. CPU достаточно.

**Deliverable:** исправленный loop, decreasing train loss, повторяемая validation-метрика и список найденных ошибок.

In [ ]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
X = torch.randn(512, 4)
y = ((X[:, 0] - .7 * X[:, 1] + .2 * X[:, 2]) > 0).long()
train_loader = DataLoader(TensorDataset(X[:400], y[:400]), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X[400:], y[400:]), batch_size=64, shuffle=False)
model = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Dropout(.2), nn.Linear(16, 2))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()


In [ ]:
# BROKEN: сделайте code review, затем замените реализацию.
def train_epoch(model, loader):
    losses = []
    model.eval()
    for xb, yb in loader:
        logits = model(xb)
        loss = criterion(logits, yb)
        optimizer.step()
        loss.backward()
        losses.append(loss.item())
    return float(np.mean(losses))

def validate(model, loader):
    model.train()
    correct = 0
    for xb, yb in loader:
        logits = model(xb)
        correct += (logits.argmax(1) == yb).sum().item()
    return correct / len(loader.dataset)


In [ ]:
# Self-check после исправления.
history = [train_epoch(model, train_loader) for _ in range(8)]
score_1 = validate(model, val_loader)
score_2 = validate(model, val_loader)
assert history[-1] < history[0], history
assert score_1 == score_2, "validation должна быть детерминирована при неизменной модели"
assert score_1 > 0.75, score_1
print({"first_loss": history[0], "last_loss": history[-1], "val_accuracy": score_1})


## Review checklist

Для каждой ошибки укажите механизм: почему `step()` до `backward()` не обновляет текущий градиент, зачем обнулять gradients, чем `train()` отличается от `eval()`, и почему validation выполняют без записи autograd graph.